# 01 · Install dependencies and verify services

In [1]:
import os
import boto3
from botocore.config import Config
import psycopg2
import mlflow
from mlflow.tracking import MlflowClient

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
S3_ENDPOINT = os.environ.get("MLFLOW_S3_ENDPOINT_URL", "http://storage:9000")
BUCKET = os.environ.get("S3_BUCKET", "mlflow-artifacts")
mlflow.set_tracking_uri(TRACKING_URI)


## Check the separate dataset database

In [2]:
conn = psycopg2.connect(
    host=os.environ.get("DATASET_POSTGRES_HOST", "dataset-postgres"),
    port=os.environ.get("DATASET_POSTGRES_PORT", "5432"),
    dbname=os.environ.get("DATASET_POSTGRES_DB", "heart_disease"),
    user=os.environ.get("DATASET_POSTGRES_USER", "dataset_user"),
    password=os.environ.get("DATASET_POSTGRES_PASSWORD", "dataset_pass"),
)
with conn, conn.cursor() as cur:
    cur.execute("SELECT current_database(), version()")
    print("Dataset DB:", cur.fetchone()[0])
conn.close()


Dataset DB: heart_disease


## Create or verify the MinIO bucket and check MLflow

In [3]:
s3 = boto3.client(
    "s3", endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID", "s3admin"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY", "s3admin123"),
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    config=Config(s3={"addressing_style": "path"}),
)
existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f"Bucket {BUCKET} created in MinIO")
else:
    print(f"Bucket {BUCKET} already exists")
client = MlflowClient()
print("MLflow URI:", TRACKING_URI)
print("Existing experiments:", [e.name for e in client.search_experiments()])


Bucket mlflow-artifacts created in MinIO
MLflow URI: http://mlflow:5000
Existing experiments: ['Default']
